# 06 — Qwen + CrafText: ручной эпизод

Тетрадка поднимает реальную среду CrafText и локальную Qwen 2.5 0.5B через Tunix. Мы вручную пройдём путь `reset → prompt → completion → decode/fallback → step → replay`, чтобы каждый слой был видим.

Из корня репозитория установите extras и заранее поместите разрешённый local snapshot в `artifacts/models/qwen25-05b-instruct`:

```bash
pyenv exec python -m uv sync --all-extras
```


In [ ]:
from pathlib import Path

import jax

from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.models.llm import LlmRequest
from tunix_craftext.env.prompts import MegaPromptRenderer, PromptContext
from tunix_craftext.artifacts.replay import ReplayArtifact, ReplayStep, save_replay
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.text_policy import DecodeMetrics, DecodedAction, decode_action_outcome
from tunix_craftext.models.tunix_adapter import QwenTunixBackend


In [ ]:
ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents) if (directory / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository.')

SNAPSHOT = ROOT / 'artifacts' / 'models' / 'qwen25-05b-instruct'
if not SNAPSHOT.is_dir():
    raise FileNotFoundError(
        f'Expected local Qwen weights at {SNAPSHOT}. Download them explicitly; the notebook never downloads weights.'
    )

config_path = ROOT / 'configs' / 'mvp' / 'tiny_craftext.yaml'
config = load_mvp_config(config_path)
runtime = build_craftext_runtime(config)
reset = runtime.adapter.reset(jax.random.PRNGKey(config.run.seed))

print('observation shape:', reset.observation.shape)
print('actions:', runtime.actions.labels)


In [ ]:
# This is the real vendored MegaPrompts template declared by the config.
# It consumes structured CrafText EnvState, not the pixel observation array.
renderer = MegaPromptRenderer(config.prompt.template)
prompt = renderer.render(
    PromptContext(
        goal='Stay alive and inspect the world.',
        observation=reset.state,
        actions=runtime.actions,
    )
)
print(prompt.text)


In [ ]:
# The full `base` template is padded by Tunix to 1024 prompt tokens; leave room for decoding.
backend = QwenTunixBackend(SNAPSHOT, cache_size=2048, seed=config.run.seed)
response = backend.complete(LlmRequest(prompt, max_new_tokens=8))

print('backend:', response.backend)
print('model:', response.model)
print('completion:', repr(response.raw_text))
print('token ids:', response.token_ids)
print('token logprobs:', response.token_logprobs)


In [ ]:
decoded, metrics = decode_action_outcome(prompt, response.raw_text)
fallback_action_id = runtime.actions.index_of('NOOP')
if decoded is not None and not bool(reset.action_mask[decoded.action_id]):
    metrics = DecodeMetrics(masked_action=1)
    decoded = None
fallback_used = decoded is None
decision = decoded or DecodedAction(
    action_id=fallback_action_id,
    label=runtime.actions.labels[fallback_action_id],
    raw_text=response.raw_text,
)

print('decode metrics:', metrics)
print('fallback used:', fallback_used)
print('environment action:', decision.action_id, decision.label)


In [ ]:
step_key = jax.random.PRNGKey(config.run.seed + 1)
transition = runtime.adapter.step(step_key, reset.state, decision.action_id)

print('reward:', float(transition.reward))
print('terminated:', bool(transition.terminated))
print('next observation shape:', transition.observation.shape)


In [ ]:
artifact = ReplayArtifact(
    config_path=str(config_path.relative_to(ROOT)),
    commit='manual-notebook',
    backend=f'{response.backend}:{response.model}',
    steps=(
        ReplayStep(
            index=0,
            prompt=prompt.text,
            raw_completion=response.raw_text,
            action_id=decision.action_id,
            action_label=decision.label,
            reward=float(transition.reward),
            terminated=bool(transition.terminated),
            invalid_format=metrics.invalid_format,
            unknown_action=metrics.unknown_action,
            masked_action=metrics.masked_action,
            action_mask=tuple(bool(value) for value in reset.action_mask),
            prompt_token_ids=response.prompt_token_ids,
            fallback_used=fallback_used,
            token_logprobs=response.token_logprobs,
            token_ids=response.token_ids,
        ),
    ),
)
output = ROOT / 'artifacts' / 'trajectories' / 'qwen-craftext-manual-replay.json'
save_replay(output, artifact)
print(output)
